# EPL — ADYm project



- Nguồn: thư viện Python `premier_league` 1.0.4 (dữ liệu tích hợp, **không nhất thiết cập nhật đến hiện tại**).
- Lấy dữ liệu bằng **object** `MatchStatistics()`.
- Chỉ giữ giải **English Premier League**, từ mùa **2017–2018** đến dữ liệu có sẵn ở mùa **2024–2025**.
- Đầu ra: 2 file CSV (trận đấu và thống kê đội) được đóng gói thành một ZIP.


## Bước 1 — Cài thư viện


In [1]:
%pip -q install "premier_league==1.0.4" pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8

## Bước 2 — Khởi tạo object
Gọi `MatchStatistics()`.

In [2]:
import pandas as pd
from premier_league import MatchStatistics

api = MatchStatistics()
print("Tổng trận của tất cả giải trong thư viện:", api.get_total_game_count())

Tổng trận của tất cả giải trong thư viện: 17945


## Bước 3 — Lấy dữ liệu từng mùa, từng vòng
`get_games_by_season(mùa, vòng)` trả về danh sách dictionary, mỗi dictionary là một trận.

- Duyệt mùa giải và vòng 1–38.
- **Chỉ lấy Premier League** để không trộn với La Liga, Serie A...
- Tách thành hai bảng: **trận đấu** và **thống kê đội–trận**.

Nếu thư viện không có dữ liệu cho một vòng, ghi nhận là thiếu chứ **không tự tạo trận**.

In [9]:
seasons = [f"{y}-{y+1}" for y in range(2017, 2025)]
match_rows, stat_rows = [], []
missing_weeks = []

for season in seasons:
    for week in range(1, 39):
        try:
            games = api.get_games_by_season(season, match_week=week)
        except ValueError as error:
            if "No games found" not in str(error):
                raise
            missing_weeks.append((season, week))
            continue

        for game in games:
            if game["league"]["name"] != "Premier League":
                continue

            match_rows.append({
                "game_id": game["id"],
                "season": game["season"],
                "match_week": game["match_week"],
                "date": game["date"],
                "home_team_id": game["home_team_id"],
                "away_team_id": game["away_team_id"],
                "home_team": game["home_team"]["name"],
                "away_team": game["away_team"]["name"],
                "home_goals": game["home_goals"],
                "away_goals": game["away_goals"],
            })
            stat_rows.extend(game.get("game_stats", []))

matches = pd.DataFrame(match_rows)
team_stats = pd.DataFrame(stat_rows)


print("Số trận EPL:", len(matches))
print("Số dòng thống kê đội–trận:", len(team_stats))
print("Số vòng thiếu hoàn toàn trong nguồn:", len(missing_weeks))
display(matches.head())
print()
display(team_stats.head())

Số trận EPL: 2970
Số dòng thống kê đội–trận: 5940
Số vòng thiếu hoàn toàn trong nguồn: 0


,game_id,season,match_week,date,home_team_id,away_team_id,home_team,away_team,home_goals,away_goals
0,e3c3ddf0,2017-2018,1,2017-08-11T19:45:00,18bb7c10,a2d435b3,Arsenal,Leicester City,4,3
1,60f6cc1d,2017-2018,1,2017-08-12T12:30:00,2abfe087,822bd0ba,Watford,Liverpool,3,3
2,2d369d17,2017-2018,1,2017-08-12T15:00:00,47c64c55,f5922ca5,Crystal Palace,Huddersfield Town,0,3
3,684f704a,2017-2018,1,2017-08-12T15:00:00,60c6b05f,4ba7cbea,West Bromwich Albion,Bournemouth,1,0
4,71b00bca,2017-2018,1,2017-08-12T15:00:00,cff3d9bb,943e8050,Chelsea,Burnley,2,3


,id,game_id,team_id,xG,xA,xAG,shots_total_FW,shots_total_MF,shots_total_DF,shots_on_target_FW,...,pens_conceded,fouls_committed_FW,fouls_committed_MF,fouls_committed_DF,fouls_drawn_FW,fouls_drawn_MF,fouls_drawn_DF,offside_FW,offside_MF,offside_DF
0,30991,e3c3ddf0,18bb7c10,3.1,1.7,2.6,10,9,1,4,...,0,3,1,3,6,2,3,5,0,0
1,30992,e3c3ddf0,a2d435b3,1.5,1.2,1.4,4,2,0,3,...,0,3,4,5,2,5,2,1,2,0
2,30993,60f6cc1d,2abfe087,2.2,0.5,0.6,4,2,3,1,...,1,9,2,2,6,0,2,3,0,0
3,30994,60f6cc1d,822bd0ba,2.7,0.6,1.8,8,2,3,2,...,0,2,3,3,4,3,5,1,0,0
4,30995,2d369d17,47c64c55,1.2,0.6,0.8,6,3,2,3,...,0,2,1,0,6,9,0,0,0,0


## Bước 4 — Kiểm tra chất lượng dữ liệu
- `isna().sum()`: đếm ô thiếu.
- `duplicated()`: đếm dòng trùng theo khóa.
- `groupby()`: đếm số trận của từng mùa, phát hiện mùa chưa đủ 380 trận.

In [4]:
print("Số trận theo mùa:")
display(matches.groupby("season").size().rename("so_tran"))

print("Trùng mã trận:", matches.duplicated("game_id").sum())
print("Trùng khóa trận–đội:", team_stats.duplicated(["game_id", "team_id"]).sum())
print("Thiếu bàn thắng:", matches[["home_goals", "away_goals"]].isna().sum().sum())
print("Thiếu save_percentage ban đầu:", team_stats["save_percentage"].isna().sum())
print("Vòng thiếu hoàn toàn:", missing_weeks)

Số trận theo mùa:


,so_tran
season,
2017-2018,380
2018-2019,380
2019-2020,380
2020-2021,380
2021-2022,380
2022-2023,380
2023-2024,380
2024-2025,310


Trùng mã trận: 0
Trùng khóa trận–đội: 0
Thiếu bàn thắng: 0
Thiếu save_percentage ban đầu: 177
Vòng thiếu hoàn toàn: []


## Bước 5 — Làm sạch
1. Bỏ trùng theo khóa của trận và thống kê.
2. Chuẩn hóa tên đội và ngày thi đấu.
3. Tỷ lệ cứu thua nằm ngoài **0–100%** là bất hợp lệ → chuyển thành `NaN`, **không** thay bằng 0.
4. Kiểm tra lại khóa và số dòng thống kê mỗi trận.

In [5]:
matches = matches.drop_duplicates("game_id").copy()
team_stats = team_stats.drop_duplicates(["game_id", "team_id"]).copy()

matches["home_team"] = matches["home_team"].str.strip()
matches["away_team"] = matches["away_team"].str.strip()
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")

team_stats["save_percentage"] = pd.to_numeric(
    team_stats["save_percentage"], errors="coerce"
)
invalid = (team_stats["save_percentage"] < 0) | (team_stats["save_percentage"] > 100)
team_stats.loc[invalid, "save_percentage"] = float("nan")

matches = matches.sort_values(["date", "game_id"]).reset_index(drop=True)
team_stats = team_stats.sort_values(["game_id", "team_id"]).reset_index(drop=True)

assert matches["date"].notna().all(), "Có ngày không đọc được"
assert matches[["home_goals", "away_goals"]].notna().all().all(), "Thiếu tỷ số"
assert matches["game_id"].is_unique, "Trùng mã trận"
assert (matches["home_team_id"] != matches["away_team_id"]).all(), "Hai đội trùng nhau"
assert team_stats.groupby("game_id").size().reindex(matches["game_id"], fill_value=0).eq(2).all(), "Có trận không đủ 2 đội thống kê"

print("Đã sửa số tỷ lệ cứu thua bất hợp lệ:", int(invalid.sum()))
print("Giá trị save_percentage thiếu sau xử lý:", int(team_stats["save_percentage"].isna().sum()))
print("Số trận giữ lại:", len(matches))

Đã sửa số tỷ lệ cứu thua bất hợp lệ: 16
Giá trị save_percentage thiếu sau xử lý: 193
Số trận giữ lại: 2970


## Bước 6 — Xuất file và tải về máy
ZIP chứa hai files CSV.

- `epl_matches_clean.csv`: một dòng cho thống kê mỗi trận (tỷ số là **target**).
- `epl_team_stats_clean.csv`: một dòng thống kê cho mỗi đội trong mỗi trận (thống kê **sau trận**).


In [7]:
from zipfile import ZipFile
from google.colab import files

matches.to_csv("epl_matches_clean.csv", index=False)
team_stats.to_csv("epl_team_stats_clean.csv", index=False)

with ZipFile("EPL_clean_data.zip", "w") as zip_file:
    zip_file.write("epl_matches_clean.csv")
    zip_file.write("epl_team_stats_clean.csv")

files.download("EPL_clean_data.zip")
print("Hoàn thành!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hoàn thành!
